# 37장 · 기종·판본·인용문으로 근거를 판정하기

“A형도 표시등을 숨길 수 있나요?”에 B형 사용법이 검색됐습니다. 두 문서 모두 표시등을 설명하므로 검색 점수가 높을 수 있습니다. 높은 순위와 질문에 답할 충분한 근거를 구분하는 판정 노드를 만듭니다.

짝 활동에는 [실습 수첩](pre-mid/documents/graph/workshop.md)을 사용합니다. 과제 아래에서 풀이를 펼칠 수 있습니다.

## 후보를 세 단계로 읽기

첫째, 기종과 적용 판본을 대조합니다. 둘째, 질문에서 요구한 동작의 답이 실제 문장에 있는지 읽습니다. 셋째, 인용하려는 문장이 원문에 그대로 있는지 확인합니다. 같은 파일에 있다는 사실만으로 같은 기능을 설명한다고 볼 수 없습니다.

3부에서 쓰던 세 PDF와 공용 검색기를 같은 폴더에서 엽니다. 교재 ZIP 안의 `pre-mid/documents/rag`, `graph`, `sources`가 함께 있어야 합니다. 실제 검색 셀은 3부 Chroma 색인·OpenAI 임베딩과 API 키를 사용합니다. 처음 실행한다면 노트북 셀에서 `%pip install "langgraph>=1,<2" "langchain-openai>=1,<2" "chromadb>=1,<2" pypdf openai python-dotenv streamlit`를 실행하고 커널을 다시 시작하세요. PDF 직접 읽기와 고정 입력 경로 실험은 키 없이 실행됩니다.

In [1]:
from pathlib import Path
import sys

# 통합 ZIP과 저장소에서 같은 자료 폴더를 찾습니다.
docs = next(p / "pre-mid" / "documents" for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "pre-mid" / "documents" / "graph").is_dir())
sys.path.insert(0, str(docs / "graph"))
sys.path.insert(0, str(docs / "rag"))
from rag_lab import MANUALS, read_pages

# 세 설명서의 모든 페이지를 실제 PDF에서 읽습니다.
pages = [page for path in MANUALS for page in read_pages(path)]
by_id = {page["id"]: page for page in pages}
from recorder_graph import matching_pages, validate_quotes

old = by_id["recorder-a-1.0.pdf#p1"]
other = by_id["recorder-b-1.0.pdf#p1"]
right = by_id["recorder-a-1.1.pdf#p1"]
candidates = [other, old, right]
print([p["id"] for p in matching_pages(candidates, "A", "1.1")])

['recorder-a-1.1.pdf#p1']


B형은 기종이 다르고 이전 A형은 판본이 다릅니다. 이 예에서는 학생이 질문의 날짜를 읽고 A·1.1을 직접 선택합니다. 일치한 후보가 빈 목록이라면 이 세 후보에서는 답을 만들 수 없습니다.

## 확인할 수 있는 문장과 확인할 수 없는 주장

A형 설명서 2쪽에는 케이블을 이용한 복사가 나옵니다. 이 문장은 “무선 전송이 되나요?”에 대한 긍정도 부정도 증명하지 않습니다. “케이블 복사는 안내됨”과 “무선 지원 여부는 확인할 수 없음”을 나눠 말하면 문서의 범위를 벗어나지 않습니다.

In [2]:
quote = "녹음하는 동안 표시등을 끄는 설정은 없습니다."
claims = [{"id": right["id"], "quote": quote},
          {"id": right["id"], "quote": "설정에서 숨김을 선택하세요."}]
checked = validate_quotes([right], claims)
print([(p["id"], p["quote"]) for p in checked])

[('recorder-a-1.1.pdf#p1', '녹음하는 동안 표시등을 끄는 설정은 없습니다.')]


첫 문장은 공백과 줄바꿈을 정리한 PDF 본문에서 찾을 수 있습니다. 둘째 문장은 A형 원문의 직접 인용이 아니어서 제외됩니다. 정확한 인용문인지 검사하는 코드가 그 문장의 의미까지 보장하지는 않습니다.

## 일단 답하거나 보류하는 그래프

연습용 판정자는 학생이 원문에서 선택한 문구를 전달합니다. choose가 근거 유무를 보고 두 경로 중 하나를 선택합니다. 아래 그래프는 판정 기준과 연결을 분리해서 시험하기 위한 것입니다.

```mermaid
flowchart TD
  J[후보 판정] --> C{근거가 있는가}
  C -->|있음| A[인용 표시]
  C -->|없음| H[답변 보류]
```

In [3]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class Check(TypedDict, total=False):
    documents: list
    model: str
    edition: str
    quotes: list
    evidence: list
    status: str

def judge(state):
    pages = matching_pages(state["documents"], state["model"], state["edition"])
    return {"evidence": validate_quotes(pages, state["quotes"])}

def choose(state):
    return "cite" if state["evidence"] else "hold"

plan = StateGraph(Check)
plan.add_node("judge", judge)
plan.add_node("cite", lambda state: {"status": "인용 표시"})
plan.add_node("hold", lambda state: {"status": "보류"})
plan.add_edge(START, "judge")
plan.add_conditional_edges("judge", choose, ["cite", "hold"])
plan.add_edge("cite", END)
plan.add_edge("hold", END)
checker = plan.compile()
case = {"documents": candidates, "model": "A", "edition": "1.1", "quotes": [claims[0]]}
print(checker.invoke(case)["status"])
print(checker.invoke({**case, "documents": [old, other]})["status"])

인용 표시
보류


두 실험의 질문 조건과 인용 제안은 같습니다. 둘째 실험에서는 그 인용이 나왔다는 출처 페이지 자체가 없으므로 hold 경로가 선택됩니다.

## 판정자에게 무엇을 요청하는가

사람이 하나씩 고르던 직접 인용을 모델에 맡길 때는 “이 문서가 관련 있나요?”라는 질문만으로 부족합니다. 기종·판본을 먼저 좁힌 뒤 원래 질문과 그 후보들의 ID·본문을 보냅니다. 모델은 원문의 직접 인용, ID, 이유를 돌려줍니다. 질문의 모든 조건에 답할 근거가 없으면 인용은 빈 목록입니다. 38장의 그래프는 이 판정을 실제 모델에 요청하고 위의 원문 확인을 한 번 더 합니다.

책에 있는 ‘녹음이 멈춥니다’라는 표현을 찾았더라도 그 앞 조건이 ‘저장 공간이 가득 차면’이라면 ‘파일 경계에서 멈추나요?’의 답으로 쓸 수 없습니다. 인용한 문장의 앞뒤 한 문장을 읽는 이유입니다.

## 모델이 맞는 ID와 조금 다른 문장을 돌려주면

판정 모델에게 원문을 그대로 인용하라고 요청해도 “설정할 수 없다”처럼 뜻이 비슷한 말로 바꿔 쓸 수 있습니다. 직접 인용 표시에는 그 차이가 중요합니다. 아래 두 문장을 번갈아 넣어 어떤 것이 원문 검사를 통과하는지 예상하세요.

In [4]:
for sentence in ["화면 밝기를 낮춰도 표시등은 켜져 있습니다.", "화면 밝기를 낮춰도 표시등은 꺼지지 않습니다."]:
    proposed = [{"id": right["id"], "quote": sentence}]
    print(sentence, bool(validate_quotes([right], proposed)))

화면 밝기를 낮춰도 표시등은 켜져 있습니다. True
화면 밝기를 낮춰도 표시등은 꺼지지 않습니다. False


둘째 문장은 뜻이 비슷해도 질문자의 직접 인용으로 표시할 수 없습니다. 답변 본문은 쉬운 말로 풀어 쓸 수 있지만, 직접 인용 칸에는 검증된 원문을 그대로 놓고 사용자가 둘을 비교할 수 있게 합니다.

## 실습 · 같은 글자를 다른 판본에서 발견하기

세 후보 중 A형 3월과 5월의 표시등 문구가 같다는 점을 확인합니다. 문구가 같아도 5월 답에 구판을 출처로 붙이면 어떤 검증에서 막히는지 실험하세요.

In [5]:
# 위 조건을 만족하는 코드를 이 셀에 작성하세요.

<details><summary>모범 예시와 풀이</summary><pre><code>wrong_edition = {&quot;id&quot;: old[&quot;id&quot;], &quot;quote&quot;: quote}
print(checker.invoke({**case, &quot;quotes&quot;: [wrong_edition]})[&quot;status&quot;])
print(checker.invoke({**case, &quot;quotes&quot;: [claims[0]]})[&quot;status&quot;])</code></pre><p>인용 ID는 기종·판본을 통과한 후보 안에서만 찾습니다. 내용이 우연히 같아도 구판은 해당 시점의 근거로 사용할 수 없습니다.</p></details>

## 실습 · 케이블 근거로 무선 지원을 판단할 수 있을까

B형 2쪽에서 파일 복사 문장을 찾아 정확히 인용하세요. “무선 전송을 지원하지 않는다”는 단정을 증명하는지 판정하고, 학생이 상담 답변 두 문장을 씁니다.

In [6]:
# 위 조건을 만족하는 코드를 이 셀에 작성하세요.

<details><summary>모범 예시와 풀이</summary><pre><code>b_page = by_id[&quot;recorder-b-1.0.pdf#p2&quot;]
line = &quot;녹음을 끝낸 뒤 케이블로 컴퓨터에 연결하세요.&quot;
print(validate_quotes([b_page], [{&quot;id&quot;: b_page[&quot;id&quot;], &quot;quote&quot;: line}])[0][&quot;quote&quot;])</code></pre><p>참고 답은 “B형 1.0판 2쪽에는 케이블로 복사하는 방법이 있습니다. 이 설명서만으로 무선 전송 지원 여부는 확인하기 어렵습니다.”입니다. 원문에 있는 문장과 질문을 해결할 수 있는 문장을 구분하세요.</p></details>

## 전이 과제 · 질문 두 조건 중 하나만 증명된 경우

상담자가 “4월 A형 파일이 몇 분마다 바뀌고 그때 표시등도 꺼지나요?”라고 묻습니다. A형 1.1판 2쪽에는 45분과 녹음 지속이 있습니다. 1쪽에는 녹음 중 표시등이 켜지고 녹음을 끝내면 꺼진다는 설명이 있습니다. 두 페이지가 필요한 질문을 인용 한 문장이 발견되었다는 이유로 끝내면 어느 조건을 놓칠까요?

질문을 학생 말로 두 부분으로 나눕니다. 각각 어느 페이지에서 어떤 문장으로 답할 수 있는지 적어 봅니다. 첫 번째 페이지만 전달받았다면 확인 가능한 부분과 추가로 확인해야 할 부분을 나누어 한 문장씩 쓰세요. 두 인용문에서 도출한 결론과 글자 그대로 적힌 인용도 구분합니다.

In [7]:
# 먼저 예상 결과를 적고, 위 상황을 재현할 셀을 작성하세요.

<details><summary>모범 예시와 풀이</summary><pre><code>wanted = [(&quot;recorder-a-1.1.pdf#p2&quot;, &quot;파일이 바뀌어도 녹음은 계속됩니다.&quot;),
          (&quot;recorder-a-1.1.pdf#p1&quot;, &quot;녹음 중에는 앞면의 빨간 표시등이 켜집니다.&quot;)]
for pid, sentence in wanted:
    checked = validate_quotes([by_id[pid]], [{&quot;id&quot;: pid, &quot;quote&quot;: sentence}])
    print(pid, bool(checked))</code></pre><p>2쪽에서 45분마다 파일이 열리고 녹음이 계속됨을 확인하고, 1쪽에서 녹음 중 표시등이 켜진다는 사실을 확인합니다. 두 문장을 연결한 설명은 직접 인용과 구분합니다. 문서가 새 파일 생성 순간의 하드웨어 점멸까지 측정한 것은 아니므로 그처럼 구체적인 동작은 추가 근거가 필요합니다.</p></details>

다음 장에서는 보류하기 전에 한 번 더 찾을 기회를 줍니다.